<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-12/AI_Modul_12.10_Praktikum_Training_Dataset_Citra.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Modul 12.10: Praktikum Pelatihan Dataset Citra dan Ekspor ONNX
**Tujuan Pembelajaran:**
1. Membangun pipeline dataset citra terpadu dengan augmentasi dinamis di PyTorch.
2. Mengimplementasikan siklus pelatihan berkecepatan tinggi dengan `torch.cuda.amp` (Automatic Mixed Precision).
3. Mengekspor model visi terlatih ke format standar industri biner ONNX.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)
print(f"PyTorch Versi: {torch.__version__}")

### Bagian 1: Pembuatan Dataset Citra Terstruktur dan Augmentasi

In [ ]:
class AgroDatasetPipeline(Dataset):
    def __init__(self, n_samples=300, is_train=True):
        self.is_train = is_train
        self.images = np.zeros((n_samples, 3, 32, 32), dtype=np.float32)
        self.labels = np.zeros(n_samples, dtype=np.int64)
        
        for i in range(n_samples):
            label = i % 3
            self.labels[i] = label
            if label == 0:   # Daun Sehat (Hijau)
                c = [0.15, 0.75, 0.20]
            elif label == 1: # Hawar Daun (Kuning Kering)
                c = [0.80, 0.70, 0.15]
            else:          # Karat Daun (Bercak Cokelat Kemerahan)
                c = [0.65, 0.25, 0.10]
            noise = np.random.normal(0, 0.05, (3, 32, 32))
            self.images[i] = np.clip(np.array(c)[:, None, None] + noise, 0.0, 1.0)
            
        self.images = torch.tensor(self.images)
        self.labels = torch.tensor(self.labels)
        
    def __len__(self):
        return len(self.labels)
        
    def __getitem__(self, idx):
        img = self.images[idx]
        # Augmentasi horizontal flip acak saat pelatihan
        if self.is_train and torch.rand(1).item() > 0.5:
            img = torch.flip(img, dims=[2])
        return img, self.labels[idx]

train_ds = AgroDatasetPipeline(n_samples=240, is_train=True)
val_ds = AgroDatasetPipeline(n_samples=60, is_train=False)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)
print(f"Pipeline Dataset Terpasang: {len(train_ds)} Train, {len(val_ds)} Val.")

### Bagian 2: Pelatihan dengan Automatic Mixed Precision (AMP)

In [ ]:
class CompactVisionModel(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(64, num_classes)
        )
    def forward(self, x):
        return self.net(x)

model = CompactVisionModel(num_classes=3)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

use_cuda = torch.cuda.is_available()
scaler = torch.cuda.amp.GradScaler(enabled=use_cuda)
device = torch.device('cuda' if use_cuda else 'cpu')
model.to(device)

epochs = 6
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        
        with torch.cuda.amp.autocast(enabled=use_cuda):
            out = model(imgs)
            loss = criterion(out, lbls)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item() * imgs.size(0)
        
    epoch_loss = running_loss / len(train_loader.dataset)
    print(f"Epoch [{epoch+1:02d}/{epochs}] - Loss: {epoch_loss:.4f} (Device: {device})")

print("[VALIDASI SUKSES] Pelatihan AMP selesai!")

### Bagian 3: Ekspor Model Terlatih ke Format Biner ONNX

In [ ]:
model.eval()
onnx_filename = 'model_bibit_sawit_vision_12_10.onnx'
dummy_x = torch.randn(1, 3, 32, 32, device=device)

torch.onnx.export(
    model,
    dummy_x,
    onnx_filename,
    export_params=True,
    opset_version=14,
    do_constant_folding=True,
    input_names=['input_tensor'],
    output_names=['class_logits'],
    dynamic_axes={'input_tensor': {0: 'batch_size'}, 'class_logits': {0: 'batch_size'}}
)

file_size_kb = os.path.getsize(onnx_filename) / 1024.0
print(f"[SUKSES] Model ONNX tersimpan di '{onnx_filename}' (Ukuran: {file_size_kb:.2f} KB)")
assert os.path.exists(onnx_filename), "File ONNX tidak ditemukan!"